# 8장 웹부록 | CPU와 GPU의 행렬 곱셈 경주

「영화로 보는 AI 리터러시」 8장 'AI의 두뇌와 심장: 반도체·나노기술·데이터센터'의 코랩 실습입니다.

코랩의 무료 GPU를 켜고, 같은 크기의 행렬 곱셈을 CPU와 GPU로 각각 해 봅니다. 행렬 크기를 키워 가며 걸린 시간을 그래프로 그리고, 몇 배 빨라지는지 확인합니다. 마지막에는 작은 신경망 하나를 두 방식으로 학습시켜 시간을 비교합니다.

**사용법**
- **먼저 GPU를 켜세요.** 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU**를 고르고 저장합니다.
- 그다음 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 무료 GPU는 사용자가 많을 때 배정되지 않을 수 있습니다. 그래도 노트북은 CPU만으로 끝까지 돌아갑니다(GPU 부분만 건너뜁니다).
- 그래프 글자는 한글이 깨지지 않도록 영어로 적었습니다.

## 0단계. GPU가 켜졌는지 확인하기

코랩에 미리 설치된 **PyTorch**라는 도구로 CPU와 GPU에 일을 시킵니다. 아래 칸이 GPU의 이름을 알려 주면 준비 끝입니다.

In [ ]:
import time, os
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    HAS_TORCH = True
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_TORCH, HAS_GPU = False, False

print("CPU 일꾼(코어) 수:", os.cpu_count())
if HAS_GPU:
    print("GPU 준비 완료:", torch.cuda.get_device_name(0))
elif HAS_TORCH:
    print("GPU가 꺼져 있습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 고른 뒤, 다시 위에서부터 실행하세요.")
    print("(지금 그대로 실행하면 CPU 부분만 진행합니다.)")
else:
    print("PyTorch가 없는 환경입니다. CPU 부분만 numpy로 진행합니다.")

## 1단계. 행렬 곱셈은 '곱하고 더하기'의 반복

실습 2처럼 4×4 행렬 두 개를 곱해 봅니다. 답의 칸 하나는 '네 번 곱하고 더하기'입니다.
중요한 점: 칸 하나하나는 **서로 기다릴 필요가 없습니다**. 그래서 일꾼이 많으면 모든 칸을 동시에 계산할 수 있습니다(병렬성, 3-1절).

In [ ]:
rng = np.random.default_rng(0)
A = rng.integers(1, 10, (4, 4))
B = rng.integers(1, 10, (4, 4))
print("A =\n", A)
print("B =\n", B)
row, col = 0, 0
terms = [f"{A[row, k]}×{B[k, col]}" for k in range(4)]
print(f"\n답의 (1행, 1열) 칸 = {' + '.join(terms)} = {sum(A[row, k] * B[k, col] for k in range(4))}")
print("A × B =\n", A @ B)

## 2단계. 한 칸씩 차례로 vs 한꺼번에

먼저 '사람 CPU' 방식입니다. 파이썬으로 답의 칸을 **하나씩 차례로** 계산합니다.
그다음 같은 계산을 numpy에게 **한꺼번에** 맡깁니다. numpy는 CPU 안의 여러 일꾼과, 한 번에 여러 숫자를 처리하는 회로를 함께 씁니다.

In [ ]:
def one_by_one(A, B):
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):              # 답의 칸을 하나씩
        for j in range(n):
            s = 0.0
            for k in range(n):      # 곱하고 더하기를 n번
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

for n in [32, 64, 128]:                     # ← 크기를 바꿔 보세요 (256은 오래 걸립니다)
    A = np.random.rand(n, n); B = np.random.rand(n, n)
    t0 = time.time(); one_by_one(A.tolist(), B.tolist()); slow = time.time() - t0
    t0 = time.time(); A @ B; fast = time.time() - t0 + 1e-9
    print(f"{n}×{n}: 한 칸씩 {slow * 1000:8.1f} ms   한꺼번에 {fast * 1000:6.2f} ms   → 약 {slow / fast:,.0f}배")

## 3단계. CPU와 GPU의 행렬 곱셈 경주

이제 본 경주입니다. 같은 크기의 행렬 곱셈을 CPU와 GPU에 각각 맡기고 시간을 잽니다.
크기 n인 행렬 곱셈은 곱하고 더하기를 약 n³번 합니다. n이 2배가 되면 일은 8배가 됩니다.

👉 **바꿔 볼 곳**: `SIZES`에 크기를 더하거나 빼 보세요. 8192는 CPU에서 10초 넘게 걸릴 수 있습니다.

In [ ]:
SIZES = [128, 256, 512, 1024, 2048, 4096]   # ← 행렬 크기

def time_matmul(n, device, repeat=3):
    """n×n 행렬 곱셈을 repeat번 해서 가장 빠른 시간(초)을 돌려줍니다."""
    if not HAS_TORCH:
        A = np.random.rand(n, n).astype(np.float32); B = np.random.rand(n, n).astype(np.float32)
        best = 1e9
        for _ in range(repeat):
            t0 = time.perf_counter(); A @ B; best = min(best, time.perf_counter() - t0)
        return best
    A = torch.rand(n, n, device=device); B = torch.rand(n, n, device=device)
    A @ B                                         # 준비 운동 (첫 번째는 느릴 수 있습니다)
    if device == "cuda": torch.cuda.synchronize()
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        A @ B
        if device == "cuda": torch.cuda.synchronize()   # GPU가 일을 끝낼 때까지 기다립니다
        best = min(best, time.perf_counter() - t0)
    return best

cpu_times, gpu_times = [], []
for n in SIZES:
    c = time_matmul(n, "cpu")
    cpu_times.append(c)
    line = f"{n:5d}×{n:<5d} CPU {c * 1000:9.2f} ms"
    if HAS_GPU:
        g = time_matmul(n, "cuda")
        gpu_times.append(g)
        line += f"   GPU {g * 1000:8.2f} ms   → GPU가 {c / g:6.1f}배 빠름"
    print(line)

In [ ]:
fig, axes = plt.subplots(1, 2 if HAS_GPU else 1, figsize=(11 if HAS_GPU else 6, 4))
ax = axes[0] if HAS_GPU else axes
ax.plot(SIZES, np.array(cpu_times) * 1000, "o-", label="CPU")
if HAS_GPU:
    ax.plot(SIZES, np.array(gpu_times) * 1000, "s-", label="GPU")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("Matrix size n (n x n)")
ax.set_ylabel("Time (ms, log scale)")
ax.set_title("Matrix multiplication time")
ax.legend()
if HAS_GPU:
    speedup = np.array(cpu_times) / np.array(gpu_times)
    axes[1].bar([str(n) for n in SIZES], speedup, color="#2ca02c")
    axes[1].set_xlabel("Matrix size n")
    axes[1].set_ylabel("GPU speed-up (x times)")
    axes[1].set_title("How many times faster is the GPU?")
plt.tight_layout()
plt.show()

flops = 2 * SIZES[-1] ** 3
print(f"{SIZES[-1]}×{SIZES[-1]} 곱셈 한 번 = 곱하기와 더하기 약 {flops / 1e8:,.0f}억 번")
print("(GFLOPS는 1초에 10억 번 계산한다는 단위입니다)")
print(f"CPU: 약 {flops / cpu_times[-1] / 1e9:,.0f} GFLOPS")
if HAS_GPU:
    print(f"GPU: 1초에 약 {flops / gpu_times[-1] / 1e9:,.0f} GFLOPS")

작은 행렬에서는 GPU가 별로 빠르지 않거나 오히려 느릴 수 있습니다. 일을 나눠 주고 모으는 준비 시간이 있기 때문입니다. 행렬이 커질수록, 즉 나눠 할 일이 많을수록 GPU의 일꾼 수천 명이 힘을 냅니다.

### 더 해 보기: 계산보다 느린 배달 (메모리 벽)

실습 2의 3회차처럼, 숫자를 '배달'하는 데도 시간이 듭니다. 행렬을 CPU 쪽 메모리에서 GPU로 옮기는 시간과, GPU에서 곱하는 시간을 비교해 봅니다(3-4절 메모리 벽).

In [ ]:
if HAS_GPU:
    n = 4096
    A_cpu = torch.rand(n, n); B_cpu = torch.rand(n, n)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    A_gpu = A_cpu.to("cuda"); B_gpu = B_cpu.to("cuda")
    torch.cuda.synchronize()
    move = time.perf_counter() - t0
    t0 = time.perf_counter()
    A_gpu @ B_gpu
    torch.cuda.synchronize()
    calc = time.perf_counter() - t0
    print(f"행렬 두 개를 GPU로 배달: {move * 1000:.1f} ms")
    print(f"GPU에서 곱셈 계산     : {calc * 1000:.1f} ms")
    print("배달이 계산만큼, 혹은 더 오래 걸릴 수 있습니다. 그래서 GPU 옆에 HBM을 붙여 배달 길을 넓힙니다.")
else:
    print("GPU가 없어서 이 칸은 건너뜁니다.")

## 4단계. 작은 신경망 하나를 두 방식으로 학습시키기

신경망 학습도 대부분 행렬 곱셈입니다(3-1절). 같은 신경망을 CPU와 GPU로 각각 학습시키고 시간을 잽니다.
데이터는 784칸짜리 '가짜 사진' 8,192장(10가지 종류)입니다. 신경망은 입력 784 → 은닉층 2,048 → 2,048 → 출력 10입니다.

👉 **바꿔 볼 곳**: `HIDDEN`(은닉층 크기), `EPOCHS`(바퀴 수). 은닉층을 키우면 GPU와 CPU의 차이가 더 벌어집니다.

In [ ]:
HIDDEN = 2048    # ← 은닉층 뉴런 수
EPOCHS = 3       # ← 학습 바퀴 수
N, D, K = 8192, 784, 10

gen = np.random.default_rng(0)
centers = gen.normal(0, 1, (K, D)).astype(np.float32)
labels = gen.integers(0, K, N)
data = centers[labels] + gen.normal(0, 3, (N, D)).astype(np.float32)   # 종류마다 조금씩 다른 가짜 사진

def train_on(device):
    torch.manual_seed(0)
    model = torch.nn.Sequential(
        torch.nn.Linear(D, HIDDEN), torch.nn.ReLU(),
        torch.nn.Linear(HIDDEN, HIDDEN), torch.nn.ReLU(),
        torch.nn.Linear(HIDDEN, K)).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    X = torch.tensor(data, device=device)
    y = torch.tensor(labels, device=device)
    losses = []
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        perm = torch.randperm(N, device=device)
        for i in range(0, N, 256):
            idx = perm[i:i + 256]
            loss = torch.nn.functional.cross_entropy(model(X[idx]), y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
    if device == "cuda": torch.cuda.synchronize()
    return time.perf_counter() - t0, losses

if HAS_TORCH:
    results = {"cpu": train_on("cpu")}
    print(f"CPU로 학습: {results['cpu'][0]:.1f}초  (마지막 손실 {results['cpu'][1][-1]:.3f})")
    if HAS_GPU:
        results["cuda"] = train_on("cuda")
        print(f"GPU로 학습: {results['cuda'][0]:.1f}초  (마지막 손실 {results['cuda'][1][-1]:.3f})")
        print(f"→ GPU가 약 {results['cpu'][0] / results['cuda'][0]:.1f}배 빠릅니다.")

    plt.figure(figsize=(8, 3.5))
    for dev, (sec, losses) in results.items():
        plt.plot(losses, label=f"{dev.upper()} ({sec:.1f} s)", alpha=0.8)
    plt.xlabel("Training step")
    plt.ylabel("Loss")
    plt.title("Same network, same result, different time")
    plt.legend()
    plt.show()
else:
    print("PyTorch가 없어 이 단계는 건너뜁니다. 코랩에서 열면 실행됩니다.")

두 손실 그래프는 거의 겹칩니다. 같은 계산을 하니 같은 답이 나옵니다. 다른 것은 걸린 시간뿐입니다.
2012년 알렉스넷이 게임용 그래픽 카드 두 장으로 학습한 것, 오늘날 데이터센터에 GPU 수만 장이 들어가는 것도 이 차이 때문입니다.

## 생각해 볼 것

1. 행렬이 작을 때는 GPU가 그다지 빠르지 않았습니다. 일꾼이 많은 것이 오히려 손해가 되는 경우는 언제일까요? 실습 2의 '사람 GPU 16명'과 연결해 생각해 보세요.
2. '더 해 보기'에서 배달 시간과 계산 시간을 비교했습니다. 메모리 벽이 왜 AI 칩의 가장 큰 숙제인지, HBM이 이를 어떻게 줄이는지 설명해 보세요.
3. GPU가 이렇게 빠르다면 CPU는 왜 여전히 필요할까요? 16명이 나눠 하기 어려운 계산의 예를 들어 보세요.